# Tutorial 7: Neural Networks for Market Timing

**Big Data in Finance** | Dr Daniele Bianchi  
**Duration:** 1 hour (+ 15 min bonus)

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Implement neural networks using sklearn's MLPRegressor
2. Understand the importance of feature scaling for neural networks
3. Tune hyperparameters (architecture, regularization, early stopping)
4. Compare neural networks with Ridge, Random Forest, and XGBoost
5. Evaluate economic performance (not just statistical accuracy)
6. Use AI assistants to debug code and optimize hyperparameters

---

## How This Tutorial Works

- **Run the code cells** in order (`Shift + Enter`)
- 🔧 **Exercise**: Short tasks to test your understanding
- 🤖 **AI Prompt**: Practice using ChatGPT/Claude for debugging, efficiency, and interpretation
- 💡 **Tip**: Helpful hints

---

## Context: Market Timing with Neural Networks

We'll compare neural networks against the methods from previous weeks (Ridge, Random Forest, XGBoost) for predicting market returns using the GWZ dataset.

**Key question:** Can neural networks capture non-linear patterns that improve predictions?

> 💡 **Note (scope):** This tutorial uses a deliberately simplified setup for a one-hour session — a single chronological train/test split and a reduced set of predictors — so its numbers will not exactly reproduce the full walk-forward results in the lecture slides and notes.

---

## Part 1: Setup and Data Preparation

In [ ]:
# =============================================================================
# Import libraries
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error

# Optional: XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    print("XGBoost not installed. Will use GradientBoostingRegressor instead.")
    XGB_AVAILABLE = False

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries loaded successfully!")

In [ ]:
# =============================================================================
# Load the GWZ dataset
# =============================================================================

df = pd.read_csv('Data_GWZ.csv')

# Parse date
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m')
df = df.set_index('DATE')

# Drop rows with missing values
df = df.dropna()

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# =============================================================================
# Prepare features and target
# =============================================================================

# Target: Market excess return
y = df['MktRf']

# Features: All predictors (lagged by 1 month to avoid look-ahead bias)
X = df.drop(columns=['MktRf']).shift(1)

# Align X and y (drop first row due to shift)
X = X.iloc[1:]
y = y.iloc[1:]

feature_names = list(X.columns)

print(f"Features: {len(feature_names)}")
print(f"Observations: {len(y)}")
print(f"Target mean: {y.mean()*100:.2f}%")
print(f"Target std: {y.std()*100:.2f}%")

### 1.1 Train/Test Split (Chronological)

💡 **Important:** For time series, we must split chronologically — no random shuffling!

In [ ]:
# =============================================================================
# Chronological train/test split
# =============================================================================

# Use first 80% for training, last 20% for testing
split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Training period: {X_train.index.min()} to {X_train.index.max()}")
print(f"Test period: {X_test.index.min()} to {X_test.index.max()}")
print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

### 1.2 Feature Scaling (Critical for Neural Networks!)

Neural networks are **very sensitive** to feature scales, unlike tree-based methods.

💡 **Rule:** Fit scaler on training data only, then transform both train and test.

In [ ]:
# =============================================================================
# Standardize features
# =============================================================================

scaler = StandardScaler()

# Fit on training data only!
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using training statistics
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names, index=X_test.index)

print("Before scaling (training set):")
print(X_train.describe().loc[['mean', 'std']].iloc[:, :5].round(4))
print("\nAfter scaling:")
print(X_train_scaled.describe().loc[['mean', 'std']].iloc[:, :5].round(4))

### 🤖 AI Prompt Exercise 1.1: Common Scaling Mistake

The code below has a data leakage bug. Ask an AI to identify it.

**Copy this prompt:**
```python
# My code for scaling features:
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X)  # Scale all data

# Then split
X_train_scaled = X_all_scaled[:split_idx]
X_test_scaled = X_all_scaled[split_idx:]

What's wrong with this approach? Why does it cause data leakage?
```

*Your notes from AI response:*



---

## Part 2: Neural Network Basics

### 2.1 A Simple Neural Network

In [ ]:
# =============================================================================
# Define helper function for out-of-sample R²
# =============================================================================

def oos_r2(y_true, y_pred, y_train_mean):
    """
    Calculate out-of-sample R² using training mean as benchmark.
    
    R²_OOS = 1 - SS_res / SS_tot
    where SS_tot uses the training mean (not test mean)
    """
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_train_mean)**2)
    return 1 - ss_res / ss_tot

# Training mean (our benchmark)
train_mean = y_train.mean()
print(f"Training mean (benchmark): {train_mean*100:.3f}%")

In [ ]:
# =============================================================================
# Fit a simple MLP with one hidden layer
# =============================================================================

# Simple architecture: 1 hidden layer with 50 neurons
mlp_simple = MLPRegressor(
    hidden_layer_sizes=(50,),    # One hidden layer with 50 neurons
    activation='relu',            # ReLU activation
    solver='adam',                # Adam optimizer
    alpha=0.01,                   # L2 regularization strength
    max_iter=500,                 # Maximum iterations
    early_stopping=True,          # Stop if validation score doesn't improve
    validation_fraction=0.2,      # Use 20% of training for validation
    n_iter_no_change=20,          # Stop after 20 iterations without improvement
    random_state=42
)

# Fit the model
mlp_simple.fit(X_train_scaled, y_train)

# Predictions
y_pred_mlp = mlp_simple.predict(X_test_scaled)

# Evaluate
r2_mlp = oos_r2(y_test.values, y_pred_mlp, train_mean)

print(f"Simple MLP Architecture: {mlp_simple.hidden_layer_sizes}")
print(f"Training iterations: {mlp_simple.n_iter_}")
print(f"Out-of-sample R²: {r2_mlp*100:.2f}%")

### 2.2 Understanding the Architecture

In [ ]:
# =============================================================================
# Count parameters in the network
# =============================================================================

n_features = X_train_scaled.shape[1]
n_hidden = 50
n_output = 1

# Parameters: weights + biases
params_layer1 = n_features * n_hidden + n_hidden  # Input to hidden
params_layer2 = n_hidden * n_output + n_output    # Hidden to output
total_params = params_layer1 + params_layer2

print(f"Network Architecture:")
print(f"  Input layer: {n_features} features")
print(f"  Hidden layer: {n_hidden} neurons")
print(f"  Output layer: {n_output} neuron")
print(f"\nParameter count:")
print(f"  Layer 1 (input→hidden): {n_features}×{n_hidden} + {n_hidden} = {params_layer1}")
print(f"  Layer 2 (hidden→output): {n_hidden}×{n_output} + {n_output} = {params_layer2}")
print(f"  Total: {total_params} parameters")
print(f"\nObservations in training: {len(X_train)}")
print(f"Parameters per observation: {total_params / len(X_train):.2f}")

### 🔧 Exercise 2.1: Why Regularization Matters

The network has ~1,350 parameters but only ~650 training observations. 
- What does this suggest about overfitting risk?
- Why is regularization (alpha parameter, early stopping) important?

*Your answer:*

*Your answer:*



### 2.3 Training Curves

In [ ]:
# =============================================================================
# Plot training curve (loss over iterations)
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(mlp_simple.loss_curve_, label='Training Loss', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss (MSE)')
ax.set_title('MLP Training Curve')
ax.legend()

# Mark where early stopping occurred
if mlp_simple.n_iter_ < 500:
    ax.axvline(x=mlp_simple.n_iter_, color='red', linestyle='--', 
               label=f'Early stopping at iter {mlp_simple.n_iter_}')
    ax.legend()

plt.tight_layout()
plt.show()

---

## Part 3: Hyperparameter Tuning

### 3.1 Key Hyperparameters

| Hyperparameter | What it controls | Typical values |
|---------------|------------------|----------------|
| `hidden_layer_sizes` | Network architecture | (50,), (100,), (50,25) |
| `alpha` | L2 regularization strength | 0.0001 to 0.1 |
| `learning_rate_init` | Initial learning rate | 0.001 to 0.01 |
| `early_stopping` | Prevent overfitting | True (usually) |

In [ ]:
# =============================================================================
# Grid search for hyperparameter tuning
# =============================================================================

# Define parameter grid (keeping it small for speed)
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32, 16)],
    'alpha': [0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01]
}

# Base model
mlp_base = MLPRegressor(
    activation='relu',
    solver='adam',
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=15,
    random_state=42
)

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Grid search
print("Running grid search (this may take a minute)...")
grid_search = GridSearchCV(
    mlp_base, param_grid, 
    cv=tscv, 
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score (neg MSE): {grid_search.best_score_:.6f}")

In [ ]:
# =============================================================================
# Evaluate best model on test set
# =============================================================================

best_mlp = grid_search.best_estimator_
y_pred_best_mlp = best_mlp.predict(X_test_scaled)
r2_best_mlp = oos_r2(y_test.values, y_pred_best_mlp, train_mean)

print(f"Best MLP Out-of-sample R²: {r2_best_mlp*100:.2f}%")

### 🤖 AI Prompt Exercise 3.1: Improve Grid Search Efficiency

Grid search can be slow. Ask an AI for alternatives.

**Copy this prompt:**
```
I'm using GridSearchCV to tune my neural network hyperparameters. It's taking a long time because I have 18 combinations to try.

param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32, 16)],
    'alpha': [0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01]
}

What alternatives to GridSearchCV could be faster?
1. What is RandomizedSearchCV and when is it better?
2. Are there any Bayesian optimization libraries I could use?
3. How do I decide between grid and random search?
```

*Your notes:*



---

## Part 4: Model Comparison

Let's compare neural networks with the methods from previous weeks.

In [ ]:
# =============================================================================
# Train benchmark models
# =============================================================================

# 1. Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
r2_ridge = oos_r2(y_test.values, y_pred_ridge, train_mean)

# 2. Random Forest
rf = RandomForestRegressor(
    n_estimators=100, 
    max_depth=4, 
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)  # RF doesn't need scaling
y_pred_rf = rf.predict(X_test)
r2_rf = oos_r2(y_test.values, y_pred_rf, train_mean)

# 3. Gradient Boosting / XGBoost
if XGB_AVAILABLE:
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    r2_xgb = oos_r2(y_test.values, y_pred_xgb, train_mean)
else:
    gb = GradientBoostingRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )
    gb.fit(X_train, y_train)
    y_pred_xgb = gb.predict(X_test)
    r2_xgb = oos_r2(y_test.values, y_pred_xgb, train_mean)

print("Models trained successfully!")

In [ ]:
# =============================================================================
# Compare OOS R² across models
# =============================================================================

results = pd.DataFrame({
    'Model': ['Historical Mean', 'Ridge', 'Random Forest', 
              'XGBoost' if XGB_AVAILABLE else 'Gradient Boosting', 
              'MLP (Simple)', 'MLP (Tuned)'],
    'OOS R² (%)': [0, r2_ridge*100, r2_rf*100, r2_xgb*100, 
                  r2_mlp*100, r2_best_mlp*100]
})

print("\n" + "="*50)
print("MODEL COMPARISON: Out-of-Sample R²")
print("="*50)
print(results.to_string(index=False))
print("="*50)

In [ ]:
# =============================================================================
# Visualize comparison
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['gray', 'steelblue', 'green', 'orange', 'purple', 'red']
bars = ax.bar(results['Model'], results['OOS R² (%)'], color=colors, edgecolor='black')

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_ylabel('Out-of-Sample R² (%)')
ax.set_title('Model Comparison: Market Return Prediction')
ax.set_xticklabels(results['Model'], rotation=30, ha='right')

# Add value labels on bars
for bar, val in zip(bars, results['OOS R² (%)']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}%', ha='center', va='bottom' if height >= 0 else 'top',
            fontsize=10)

plt.tight_layout()
plt.show()

### 🔧 Exercise 4.1: Interpret the Results

Based on the comparison:
1. Which model performs best? Is the difference large or small?
2. Did neural networks outperform simpler methods? Why might that be?
3. What does a negative R² mean?

*Your interpretation:*

*Your interpretation:*



---

## Part 5: Economic Performance

Statistical accuracy (R²) doesn't always translate to economic performance!

In [ ]:
# =============================================================================
# Calculate portfolio returns based on predictions
# =============================================================================

def calculate_portfolio_returns(y_actual, y_pred, gamma=5, weight_bounds=(-0.5, 1.5)):
    """
    Calculate portfolio returns using mean-variance weights.

    w_t = (1/gamma) * (predicted_return / variance)
    Weights clipped to bounds.
    """
    # Use rolling variance estimate (60-month window)
    variance = y_actual.rolling(60).var().shift(1).fillna(y_actual.var())

    # Calculate weights
    weights = (1/gamma) * (y_pred / variance.values)
    weights = np.clip(weights, weight_bounds[0], weight_bounds[1])

    # Portfolio returns (excess; the (1 - w) allocation to the risk-free asset earns zero excess)
    portfolio_returns = weights * y_actual.values

    return portfolio_returns, weights

# Store all predictions. The "Historical Mean" benchmark uses a constant
# (training-mean) forecast -- the standard, hard-to-beat benchmark from the lecture.
predictions = {
    'Ridge': y_pred_ridge,
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb,
    'MLP (Tuned)': y_pred_best_mlp,
    'Historical Mean': np.ones(len(y_test)) * train_mean
}

In [ ]:
# =============================================================================
# Calculate economic metrics for each model
# =============================================================================

economic_results = []

for model_name, preds in predictions.items():
    port_returns, weights = calculate_portfolio_returns(y_test, preds)
    
    # Annualized metrics
    ann_return = port_returns.mean() * 12 * 100
    ann_vol = port_returns.std() * np.sqrt(12) * 100
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    economic_results.append({
        'Model': model_name,
        'Ann. Return (%)': ann_return,
        'Ann. Vol (%)': ann_vol,
        'Sharpe Ratio': sharpe,
        'Avg Weight': weights.mean()
    })

econ_df = pd.DataFrame(economic_results)
print("\n" + "="*70)
print("ECONOMIC PERFORMANCE COMPARISON")
print("="*70)
print(econ_df.round(2).to_string(index=False))
print("="*70)

In [ ]:
# =============================================================================
# Plot cumulative returns
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

colors_dict = {'Ridge': 'steelblue', 'Random Forest': 'green',
               'XGBoost': 'orange', 'MLP (Tuned)': 'red', 'Historical Mean': 'black'}

for model_name, preds in predictions.items():
    port_returns, _ = calculate_portfolio_returns(y_test, preds)
    cum_returns = (1 + port_returns).cumprod()
    ax.plot(y_test.index, cum_returns, label=model_name,
            color=colors_dict[model_name], linewidth=2 if model_name in ['MLP (Tuned)', 'Historical Mean'] else 1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.set_title('Cumulative Portfolio Returns by Model')
ax.legend()
ax.axhline(y=1, color='gray', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

### 🤖 AI Prompt Exercise 5.1: Explain the Disconnect

**Copy this prompt:**
```
In my market timing exercise, I found that:
- Ridge regression has OOS R² of 1.2% but Sharpe ratio of 0.65
- MLP has OOS R² of 0.8% but Sharpe ratio of 0.58

Why might a model with lower R² have a higher Sharpe ratio? What does this tell us about evaluating ML models for finance applications?
```

*Your notes:*



---

## Part 6: When Do Neural Networks Help?

Based on the lecture and your results, let's summarize:

In [ ]:
# =============================================================================
# Summary table: When to use each method
# =============================================================================

summary = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                    WHEN TO USE EACH METHOD                                   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║ METHOD          │ STRENGTHS                    │ BEST FOR                    ║
╠═════════════════╪══════════════════════════════╪═════════════════════════════╣
║ Ridge/Lasso     │ Interpretable, fast, robust  │ Few features, need interp.  ║
║                 │ Works well with limited data │ Regulatory requirements     ║
╠═════════════════╪══════════════════════════════╪═════════════════════════════╣
║ Random Forest   │ Handles interactions         │ Many features, nonlinearity ║
║                 │ Robust to hyperparameters    │ Feature importance needed   ║
╠═════════════════╪══════════════════════════════╪═════════════════════════════╣
║ XGBoost         │ Often best accuracy          │ Competitions, large data    ║
║                 │ Feature importance, speed    │ Careful tuning available    ║
╠═════════════════╪══════════════════════════════╪═════════════════════════════╣
║ Neural Network  │ Smooth nonlinear functions   │ Very large datasets         ║
║                 │ Universal approximation      │ Complex patterns exist      ║
║                 │ Lower turnover (smooth)      │ Smooth predictions needed   ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

---

## Key Takeaways

1. **Feature scaling is critical** for neural networks
   - Always standardize features
   - Fit scaler on training data only

2. **Regularization prevents overfitting**
   - Use early stopping
   - L2 regularization (alpha parameter)
   - Keep architecture simple (1-2 hidden layers)

3. **Neural networks are not always better**
   - For small datasets, simpler methods often win
   - Returns prediction is a low signal-to-noise problem
   - Tree methods capture interactions without explicit modeling

4. **Statistical vs. Economic performance can differ**
   - Higher R² doesn't guarantee higher Sharpe
   - Consider turnover and transaction costs
   - Evaluate on economically meaningful metrics

5. **Neural networks shine when:**
   - Large datasets (cross-section of stocks)
   - Complex, smooth nonlinear patterns exist
   - You have computational resources for tuning

---

## Course Summary

Over 7 weeks, you've learned:
- **Week 1:** ML foundations, bias-variance tradeoff
- **Week 2:** Regularized regression (Ridge, Lasso)
- **Week 3:** Tree-based methods (RF, XGBoost)
- **Week 4:** Classification for credit risk
- **Week 5:** Unsupervised learning (PCA, clustering)
- **Week 6:** Model interpretability (SHAP)
- **Week 7:** Neural networks

**The key insight:** There is no universally best method. Choose based on your data, problem, and constraints!

---

## 🔧 Bonus Section (If Time Permits)

### Bonus 1: Deeper Network Architecture

In [ ]:
# Try a deeper network (2 hidden layers)
mlp_deep = MLPRegressor(
    hidden_layer_sizes=(64, 32),  # Two hidden layers
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=20,
    random_state=42
)

mlp_deep.fit(X_train_scaled, y_train)
y_pred_deep = mlp_deep.predict(X_test_scaled)
r2_deep = oos_r2(y_test.values, y_pred_deep, train_mean)

print(f"Deep MLP (64, 32) OOS R²: {r2_deep*100:.2f}%")
print(f"Compare to simple MLP (50,): {r2_mlp*100:.2f}%")
print(f"\nDoes deeper = better? {r2_deep > r2_mlp}")

### Bonus 2: Prediction Smoothness Comparison

In [ ]:
# =============================================================================
# Compare prediction volatility (smoothness)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

models_to_plot = [
    ('Ridge', y_pred_ridge),
    ('Random Forest', y_pred_rf),
    ('XGBoost', y_pred_xgb),
    ('MLP (Tuned)', y_pred_best_mlp)
]

for ax, (name, preds) in zip(axes.flatten(), models_to_plot):
    ax.plot(y_test.index, preds * 100, linewidth=1, alpha=0.8)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title(f'{name} Predictions')
    ax.set_ylabel('Predicted Return (%)')
    
    # Calculate turnover (changes in prediction)
    turnover = np.abs(np.diff(preds)).mean()
    ax.text(0.02, 0.98, f'Avg |Δ|: {turnover*100:.3f}%', 
            transform=ax.transAxes, va='top', fontsize=10)

plt.tight_layout()
plt.show()

print("\nPrediction Turnover (lower = smoother):")
for name, preds in models_to_plot:
    turnover = np.abs(np.diff(preds)).mean()
    print(f"  {name}: {turnover*100:.4f}%")

### 🤖 Bonus AI Exercise: Complete Model Selection Flowchart

Ask an AI:
```
Create a decision flowchart for choosing between Ridge, Random Forest, XGBoost, and Neural Networks for a financial prediction problem. 

The flowchart should consider:
1. Dataset size (small vs large)
2. Need for interpretability (yes/no)
3. Expected nonlinearity (low/medium/high)
4. Computational resources (limited/abundant)
5. Regulatory requirements (strict/flexible)

Format as a text-based decision tree.
```

*Paste the flowchart here:*

